In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import time

# --- 配置 ---
MAX_PAGES = 5   # 只测试5页
URL = "https://hk.eastmoney.com/buyback.html?code=&sdate=2015-01-01&edate=2026-02-04"

# 启动浏览器
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)
driver.get(URL)

print(">>> 正在初始化，等待页面加载 8 秒...")
time.sleep(8)  # 给足时间加载

all_data = []

for page in range(1, MAX_PAGES + 1):
    print(f"\n[调试] 正在处理第 {page} 页...")
    
    # --- 调试步骤 1: 寻找表格 ---
    # 我们不指定ID，而是把页面上所有的表格都找出来，看看哪一个是真的
    tables = driver.find_elements(By.TAG_NAME, "table")
    print(f"[调试] 页面上共发现 {len(tables)} 个表格")
    
    target_table = None
    
    for i, tbl in enumerate(tables):
        # 统计每个表格有多少行
        rows = tbl.find_elements(By.TAG_NAME, "tr")
        print(f"  - 表格 {i}: 共有 {len(rows)} 行")
        
        # 假设数据表格行数肯定大于 10，我们以此为标准找到目标表格
        if len(rows) > 10:
            target_table = tbl
            print(f"  -> 锁定表格 {i} 为目标表格！")
            break
            
    if target_table is None:
        print("[错误] 未找到包含数据的表格！可能数据还未加载，或被iframe包裹。")
        # 如果找不到表格，继续尝试下一页也没意义，但为了测试翻页逻辑暂时不break
    else:
        # --- 调试步骤 2: 提取行数据 ---
        rows = target_table.find_elements(By.TAG_NAME, "tr")
        page_data_count = 0
        
        for r_index, row in enumerate(rows):
            # 这里的 text 获取的是整个 tr 的文本，如果不为空说明抓到了
            txt = row.text
            if not txt.strip():
                continue # 跳过空行
            
            # 提取列
            cols = row.find_elements(By.TAG_NAME, "td")
            if len(cols) == 0:
                # 可能是表头 th
                continue
                
            # 打印第一条有效数据来看看格式
            if page_data_count == 0:
                print(f"  [数据样本] 第一行内容: {[c.text for c in cols]}")
            
            # 保存数据
            row_data = [c.text for c in cols]
            all_data.append(row_data)
            page_data_count += 1
            
        print(f"[成功] 第 {page} 页共提取到 {page_data_count} 条有效数据")

    # --- 调试步骤 3: 翻页 ---
    if page < MAX_PAGES:
        try:
            next_btn = driver.find_element(By.XPATH, "//a[contains(text(), '下一页')]")
            driver.execute_script("arguments[0].click();", next_btn)
            print("[操作] 已点击下一页，等待数据加载...")
            time.sleep(5) # 必须等待
        except Exception as e:
            print(f"[错误] 翻页失败: {e}")
            break

# --- 保存结果 ---
print(f"\n>>> 调试结束，共抓取 {len(all_data)} 条数据")

if all_data:
    # 自动根据列数生成表头，防止列数对不上报错
    col_count = len(all_data[0])
    columns = [f"列{i+1}" for i in range(col_count)]
    
    df = pd.DataFrame(all_data, columns=columns)
    df.to_excel("调试结果_前5页.xlsx", index=False)
    print("文件已保存为：调试结果_前5页.xlsx，请打开查看内容是否正确。")
else:
    print("!!! 警告：没有抓取到任何数据，请检查上方日志中的[错误]信息。")

driver.quit()

>>> 正在初始化，等待页面加载 8 秒...

[调试] 正在处理第 1 页...
[调试] 页面上共发现 1 个表格
  - 表格 0: 共有 51 行
  -> 锁定表格 0 为目标表格！
  [数据样本] 第一行内容: ['序号', '股票代码', '股票名称', '回购数量(股)', '最高回购价', '最低回购价', '回购平均价', '回购总额(港元)', '日期']
[成功] 第 1 页共提取到 51 条有效数据
[操作] 已点击下一页，等待数据加载...

[调试] 正在处理第 2 页...
[调试] 页面上共发现 1 个表格
  - 表格 0: 共有 51 行
  -> 锁定表格 0 为目标表格！
  [数据样本] 第一行内容: ['序号', '股票代码', '股票名称', '回购数量(股)', '最高回购价', '最低回购价', '回购平均价', '回购总额(港元)', '日期']
[成功] 第 2 页共提取到 51 条有效数据
[操作] 已点击下一页，等待数据加载...

[调试] 正在处理第 3 页...
[调试] 页面上共发现 1 个表格
  - 表格 0: 共有 51 行
  -> 锁定表格 0 为目标表格！
  [数据样本] 第一行内容: ['序号', '股票代码', '股票名称', '回购数量(股)', '最高回购价', '最低回购价', '回购平均价', '回购总额(港元)', '日期']
[成功] 第 3 页共提取到 51 条有效数据
[操作] 已点击下一页，等待数据加载...

[调试] 正在处理第 4 页...
[调试] 页面上共发现 1 个表格
  - 表格 0: 共有 51 行
  -> 锁定表格 0 为目标表格！
  [数据样本] 第一行内容: ['序号', '股票代码', '股票名称', '回购数量(股)', '最高回购价', '最低回购价', '回购平均价', '回购总额(港元)', '日期']
[成功] 第 4 页共提取到 51 条有效数据
[操作] 已点击下一页，等待数据加载...

[调试] 正在处理第 5 页...
[调试] 页面上共发现 1 个表格
  - 表格 0: 共有 51 行
  -> 锁定表格 0 为目标表格！
  [数据样本] 第一行内容: ['序号', '股票代码', '股票名称', '回购

调整maxpage = 1300（实际网页共1259页）

得到“调试结果”，以及完整版本的“raw”
